# Configure a Feature Group with both online and offline tiers and prove the online side returns under 10 ms while the offline side returns full history

A real-time fraud-scoring service needs sub-10 ms feature lookups at inference and a 90-day historical view for monthly retraining, and the team architect is unsure how SageMaker Feature Store splits those reads. The architect was thinking about building two separate stores. Feature Store has both built in. You have one session to prove the online tier is fast enough for the scoring service and the offline tier is queryable enough for retraining, both from one Feature Group definition.

You will ship one artifact in this lab:

- A single SageMaker Feature Group named `labrun-feature-store-online-offline-roundtrip-transactions` with both an online Standard tier (single-record `GetRecord` lookups at single-digit milliseconds) and an offline S3-backed tier (full versioned history queryable via Athena and the auto-created Glue Data Catalog table).

The Feature Group ingests 100 synthetic transaction records covering 10 distinct customers via `put_record` calls. The online tier holds only the latest record per customer; the offline tier holds every version ever ingested. The comparison metric that ships back to the platform is the measured online p95 latency across 100 reads next to the row count Athena returns from the offline tier.

**Time.** Plan for about 60 minutes hands-on. The Feature Group itself provisions in 60 to 90 seconds; the offline-tier propagation wait is the long pole at 5 to 15 minutes between PutRecord and Athena visibility. Budget 120 minutes if Athena propagation runs at the slow end.

**Cost.** This lab costs between $0.01 and $0.05 per session if cleanup tears the Feature Group within the session window. Feature Store online Standard tier bills $0.45/GB-month against a 5 GiB-month floor (prorated to about half a cent for a 90-minute session at 5 GiB) plus per-request units (100 writes plus 110 reads is well under a penny). Offline tier S3 storage is fractions of a penny at the 50 KB volume this lab writes. Athena is $5/TB scanned; the two queries scan less than 1 MB each, so fractions of a penny.

**The cost trap.** The online Standard tier bills against a 5 GiB-month floor even when you store 100 records that weigh 50 KB. The floor is what makes this lab cost a real penny instead of fractions of one. The bigger trap waits if you forget to delete the Feature Group: $0.45/GB-month at the 5 GiB floor is $2.25/month for as long as the group exists. The In-Memory tier is the catastrophic version of the same trap at $0.233/GB-hour and the same 5 GiB floor, which works out to $839/month if forgotten. This lab uses Standard exclusively; the reflection MCQ surfaces the distinction.

The cleanup cell deletes the Feature Group via `delete_feature_group` and waits for `describe_feature_group` to return `ResourceNotFound` before declaring the lab clean. Online-tier teardown takes 1 to 2 minutes after the delete call returns.

This lab maps to AWS MLA-C01 Domain 1 (Data Preparation for ML, 28%) on the Feature Store online and offline tier task statements.


In [ ]:
# NBVAL_SKIP
# Install labrun-checks. Pinned to a specific version per LAB-CREATION-BLUEPRINT.md
# build rules. Never use unpinned installs.

!pip install --quiet labrun-checks==0.6.0

In [ ]:
# NBVAL_SKIP
# Imports and per-lab constants. Resource names follow the
# labrun-{lab-slug}-{descriptor} pattern from LAB-CREATION-BLUEPRINT.md
# section 12. The full slug is 46 characters; appending a 12-digit account
# id keeps the bucket name at 58 characters, comfortably under the 63-char
# S3 limit, so this lab uses the full slug for clarity.

import atexit
import getpass
import json
import random
import sys
import time

import boto3
from botocore.exceptions import ClientError

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CleanupResource,
)

LAB_ID = "feature-store-online-offline-roundtrip"
LAB_TAG_KEY = "labrun:lab-slug"
LAB_TAG_VALUE = LAB_ID
REGION = "us-east-1"

# Resource names.
FG_NAME = f"labrun-{LAB_ID}-transactions"
FG_ROLE_NAME = f"labrun-{LAB_ID}-role"
FG_INLINE_POLICY_NAME = "labrun-mla-lab03-fg-policy"

# Bucket name, role ARN, and Glue catalog references resolve after STS returns.
BUCKET_NAME = None
FG_ROLE_ARN = None
ACCOUNT_ID = None

# Captured at runtime so checkpoint 5 can surface the comparison metric.
ONLINE_P95_MS = None
OFFLINE_ROW_COUNT = None
GLUE_DATABASE_NAME = None
GLUE_TABLE_NAME = None


In [ ]:
# NBVAL_SKIP
# Register the session, validate AWS credentials via STS GetCallerIdentity,
# resolve account-derived names. This cell must succeed before the manifest
# cell creates anything per LAB-CREATION-BLUEPRINT section 15.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
aws_access_key_id = getpass.getpass("AWS_ACCESS_KEY_ID: ")
aws_secret_access_key = getpass.getpass("AWS_SECRET_ACCESS_KEY: ")
aws_session_token = getpass.getpass(
    "AWS_SESSION_TOKEN (leave blank for long-lived credentials): "
).strip()
region_input = input(f"AWS region (default {REGION}): ").strip()
if region_input and region_input != REGION:
    print(f"Region {region_input!r} is not supported for this lab.")
    print(f"MLA Batch 1 labs run in {REGION} (N. Virginia).")
    print("Re-run this cell and accept the default.")
    raise SystemExit(1)

AWS_CREDENTIALS = {
    "aws_access_key_id": aws_access_key_id,
    "aws_secret_access_key": aws_secret_access_key,
    "region": REGION,
}
if aws_session_token:
    AWS_CREDENTIALS["aws_session_token"] = aws_session_token

sts = boto3.client(
    "sts",
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    aws_session_token=aws_session_token or None,
    region_name=REGION,
)
try:
    identity = sts.get_caller_identity()
except ClientError as e:
    print("Credentials are missing or expired. STS GetCallerIdentity failed:")
    print(f"  {e}")
    print()
    print("Refresh your AWS credentials and re-run this cell.")
    raise SystemExit(1)

ACCOUNT_ID = identity["Account"]
CALLER_ARN = identity["Arn"]
print(f"Credentials validated. Account: {ACCOUNT_ID}")
print(f"Caller ARN: {CALLER_ARN}")
print(f"Region: {REGION}")

BUCKET_NAME = f"labrun-{LAB_ID}-{ACCOUNT_ID}"
FG_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/{FG_ROLE_NAME}"
print(f"Offline-tier bucket: {BUCKET_NAME}")
print(f"Feature Group role ARN: {FG_ROLE_ARN}")
print(f"Feature Group name: {FG_NAME}")

register(session_token=session_token, lab_id=LAB_ID, credentials=AWS_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")


In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, orphan scan.
# Manifest is module-level and in reverse-creation order. The Feature Group
# is the critical resource because its online Standard tier bills $0.45/GB-month
# against a 5 GiB-month floor; even at the lab's 100-record volume the floor
# applies for as long as the group exists.
#
# labrun-checks 0.6.0 does not have an adapter for sagemaker_feature_group.
# The cleanup cell tears the Feature Group down manually BEFORE run_cleanup
# walks the manifest. The manifest below contains only adapter-supported
# types (iam_role, s3_bucket). The orphan scan still picks up any tagged
# SageMaker Feature Group via Resource Groups Tagging API.

CLEANUP_MANIFEST = [
    CleanupResource(
        type="iam_role",
        id=FG_ROLE_NAME,
        region=REGION,
        cli_delete_command=f"aws iam delete-role --role-name {FG_ROLE_NAME}",
    ),
    CleanupResource(
        type="s3_bucket",
        id=BUCKET_NAME,
        region=REGION,
        cli_delete_command=f"aws s3 rb s3://{BUCKET_NAME} --force",
    ),
]


def _atexit_cleanup() -> None:
    try:
        run_cleanup(CLEANUP_MANIFEST)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans() -> list:
    client = boto3.client(
        "resourcegroupstaggingapi",
        aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
        aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
        aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
        region_name=REGION,
    )
    arns = []
    paginator = client.get_paginator("get_resources")
    for page in paginator.paginate(
        TagFilters=[{"Key": LAB_TAG_KEY, "Values": [LAB_TAG_VALUE]}],
    ):
        for item in page.get("ResourceTagMappingList", []):
            arns.append(item.get("ResourceARN", ""))
    return arns


orphans = scan_for_orphans()
if orphans:
    print(f"BLOCKED: existing resources tagged labrun:lab-slug={LAB_TAG_VALUE} were found:")
    for arn in orphans:
        print("  -", arn)
    print()
    print("These are leftovers from a previous run of this lab. Run the cleanup")
    print("cell at the bottom of this notebook first, or remove them manually with")
    print("the AWS CLI commands the cleanup cell prints, then re-run setup.")
    raise SystemExit(1)

print("No prior orphans found. Safe to create resources for this session.")


## Task 1: Create the offline-tier S3 bucket and the IAM role Feature Store assumes at create time

Feature Store needs an IAM role at `create_feature_group` time. The role is what Feature Store assumes when it writes records to the offline-tier S3 prefix and when it creates the Glue Data Catalog table for the offline tier. Without that role, `create_feature_group` returns a directional error about missing permissions.

Build it in this order:

1. Create `labrun-feature-store-online-offline-roundtrip-{your-account-id}` and tag it `labrun:lab-slug=feature-store-online-offline-roundtrip`. This bucket holds the offline-tier records under the `feature-store-offline/` prefix and the Athena query results under `athena-results/`.
2. Create the IAM role `labrun-feature-store-online-offline-roundtrip-role` with a trust policy that lets `sagemaker.amazonaws.com` assume it.
3. Attach an inline policy granting S3 read, write, list, and delete on the lab bucket plus Glue Data Catalog database, table, and partition permissions so Feature Store can create and read the auto-generated Athena table.

The IAM role takes about 10 seconds to propagate after `put_role_policy` returns. The cell sleeps for that before yielding to Task 2.


In [ ]:
# NBVAL_SKIP
# Task 1: Create the offline-tier bucket and the Feature Store IAM role.

s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
iam = boto3.client(
    "iam",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# YOUR CODE: Create the bucket using s3.create_bucket(Bucket=BUCKET_NAME).
# us-east-1 rejects LocationConstraint in create_bucket; other regions
# require CreateBucketConfiguration={"LocationConstraint": REGION}.

s3.put_bucket_tagging(
    Bucket=BUCKET_NAME,
    Tagging={"TagSet": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]},
)
print(f"Bucket created and tagged: {BUCKET_NAME}")

trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": "sagemaker.amazonaws.com"},
            "Action": "sts:AssumeRole",
        }
    ],
}

try:
    # YOUR CODE: Create the role using iam.create_role() with
    # RoleName=FG_ROLE_NAME, AssumeRolePolicyDocument=json.dumps(trust_policy),
    # Description="labrun mla lab 03 feature group role", and
    # Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}].
    print(f"Created role: {FG_ROLE_NAME}")
except ClientError as e:
    if e.response["Error"]["Code"] == "EntityAlreadyExists":
        print(f"Role {FG_ROLE_NAME} already exists, reusing.")
    else:
        raise

inline_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "OfflineTierBucketAccess",
            "Effect": "Allow",
            "Action": [
                "s3:GetObject",
                "s3:PutObject",
                "s3:DeleteObject",
                "s3:ListBucket",
                "s3:GetBucketAcl",
                "s3:PutObjectAcl",
            ],
            "Resource": [
                f"arn:aws:s3:::{BUCKET_NAME}",
                f"arn:aws:s3:::{BUCKET_NAME}/*",
            ],
        },
        {
            "Sid": "GlueDataCatalogAccess",
            "Effect": "Allow",
            "Action": [
                "glue:GetDatabase",
                "glue:CreateDatabase",
                "glue:CreateTable",
                "glue:GetTable",
                "glue:UpdateTable",
                "glue:GetPartition",
                "glue:GetPartitions",
                "glue:CreatePartition",
                "glue:BatchCreatePartition",
            ],
            "Resource": "*",
        },
    ],
}

# YOUR CODE: Attach the inline policy using iam.put_role_policy() with
# RoleName=FG_ROLE_NAME, PolicyName=FG_INLINE_POLICY_NAME, and
# PolicyDocument=json.dumps(inline_policy).

print(f"Attached inline policy {FG_INLINE_POLICY_NAME} to {FG_ROLE_NAME}")

# IAM propagation. create_feature_group reads the role through AssumeRole, so
# give the new role 10 seconds to land.
print("Your IAM role is stuck in traffic, give it 10 seconds...")
time.sleep(10)
print("Done.")


In [ ]:
# NBVAL_SKIP
# Checkpoint 1: offline-tier bucket exists with the lab tag and the IAM role
# exists with the inline policy attached.

def checkpoint_1(session):
    from labrun_checks import CheckpointResult
    try:
        s3_client = boto3.client(
            "s3",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        iam_client = boto3.client(
            "iam",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        try:
            s3_client.head_bucket(Bucket=BUCKET_NAME)
        except ClientError as e:
            code_str = e.response["Error"]["Code"]
            if code_str in ("404", "NoSuchBucket"):
                return CheckpointResult(
                    status="fail",
                    error_reason=f"Bucket {BUCKET_NAME} does not exist.",
                )
            return CheckpointResult(status="error", error_reason=str(e))

        try:
            tag_resp = s3_client.get_bucket_tagging(Bucket=BUCKET_NAME)
            tags = {t["Key"]: t["Value"] for t in tag_resp.get("TagSet", [])}
        except ClientError as e:
            if e.response["Error"]["Code"] == "NoSuchTagSet":
                tags = {}
            else:
                return CheckpointResult(status="error", error_reason=str(e))
        if tags.get(LAB_TAG_KEY) != LAB_TAG_VALUE:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Bucket {BUCKET_NAME} is missing tag {LAB_TAG_KEY}={LAB_TAG_VALUE}. "
                    f"Found tags: {tags}"
                ),
            )

        try:
            role_resp = iam_client.get_role(RoleName=FG_ROLE_NAME)
        except ClientError as e:
            if e.response["Error"]["Code"] == "NoSuchEntity":
                return CheckpointResult(
                    status="fail",
                    error_reason=f"Role {FG_ROLE_NAME} does not exist.",
                )
            return CheckpointResult(status="error", error_reason=str(e))

        trust = role_resp["Role"]["AssumeRolePolicyDocument"]
        services = []
        for stmt in trust.get("Statement", []):
            p = stmt.get("Principal", {})
            svc = p.get("Service")
            if isinstance(svc, str):
                services.append(svc)
            elif isinstance(svc, list):
                services.extend(svc)
        if "sagemaker.amazonaws.com" not in services:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Trust policy on {FG_ROLE_NAME} does not include Service principal "
                    f"sagemaker.amazonaws.com. Found services: {services}."
                ),
            )

        inline = iam_client.list_role_policies(RoleName=FG_ROLE_NAME)
        names = inline.get("PolicyNames", [])
        if not names:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Role {FG_ROLE_NAME} has no inline policy attached. The role needs S3 "
                    f"access on the offline-tier bucket and Glue Data Catalog permissions."
                ),
            )
        doc_resp = iam_client.get_role_policy(RoleName=FG_ROLE_NAME, PolicyName=names[0])
        doc = doc_resp["PolicyDocument"]

        s3_wildcards = {"s3:*", "*"}
        glue_wildcards = {"glue:*", "*"}
        required_s3 = {"s3:GetObject", "s3:PutObject", "s3:ListBucket"}
        bucket_root = f"arn:aws:s3:::{BUCKET_NAME}"
        bucket_objects = f"arn:aws:s3:::{BUCKET_NAME}/*"

        s3_ok = False
        glue_ok = False
        for stmt in doc.get("Statement", []):
            if stmt.get("Effect") != "Allow":
                continue
            actions = stmt.get("Action", [])
            if isinstance(actions, str):
                actions = [actions]
            action_set = set(actions)
            resources = stmt.get("Resource", [])
            if isinstance(resources, str):
                resources = [resources]
            resource_set = set(resources)

            covers_s3 = required_s3.issubset(action_set) or bool(action_set & s3_wildcards)
            if covers_s3 and (
                bucket_root in resource_set
                or bucket_objects in resource_set
                or "*" in resource_set
            ):
                s3_ok = True

            covers_glue = "glue:CreateTable" in action_set or bool(action_set & glue_wildcards)
            if covers_glue:
                glue_ok = True

        if not s3_ok:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Inline policy is missing S3 access on {bucket_root!r}. Feature Store "
                    f"needs s3:GetObject, s3:PutObject, and s3:ListBucket on the offline-tier "
                    f"bucket so it can write records and the auto-created Glue table."
                ),
            )
        if not glue_ok:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Inline policy is missing Glue Data Catalog permissions. Feature Store "
                    f"creates a Glue table for the offline tier and needs glue:CreateTable, "
                    f"glue:GetTable, and friends."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(1, checkpoint_1)


<details><summary>Hint 1 (nudge)</summary>

Two things to ship: the bucket and the role. The checkpoint tells you whether a bucket is missing, untagged, the role is missing, the trust policy is wrong, or the inline policy lacks S3 or Glue permissions. Read the failure message before peeking further.

</details>

<details><summary>Hint 2 (direction)</summary>

`s3.create_bucket(Bucket=BUCKET_NAME)` with no `CreateBucketConfiguration` works for `us-east-1`; other regions need `CreateBucketConfiguration={"LocationConstraint": REGION}`. The `put_bucket_tagging` call is already wired in the cell. For the IAM role, `iam.create_role` takes Tags in the form `[{"Key": ..., "Value": ...}]`. The trust policy with `Principal.Service: sagemaker.amazonaws.com` is already built; you just need to pass it. The inline policy is also pre-built; `iam.put_role_policy(RoleName=FG_ROLE_NAME, PolicyName=FG_INLINE_POLICY_NAME, PolicyDocument=json.dumps(inline_policy))` is the call.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 1 (the three filled-in lines):

```python
s3.create_bucket(Bucket=BUCKET_NAME)

iam.create_role(
    RoleName=FG_ROLE_NAME,
    AssumeRolePolicyDocument=json.dumps(trust_policy),
    Description="labrun mla lab 03 feature group role",
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)

iam.put_role_policy(
    RoleName=FG_ROLE_NAME,
    PolicyName=FG_INLINE_POLICY_NAME,
    PolicyDocument=json.dumps(inline_policy),
)
```

</details>

**Wallet check.** One empty bucket, one IAM role, one inline policy, and a 10-second sleep. S3 control plane plus IAM at this scale sit inside the always-free tier. Damage so far: about $0.00. Your coffee this morning cost infinitely more.

## Task 2: Create the Feature Group with online Standard tier and offline S3 tier configured

This is where the dual-tier configuration lives. Three pieces have to line up:

1. The schema: 6 FeatureDefinitions covering `customer_id` (String, the RecordIdentifier), `event_time` (Fractional, the EventTime feature), `transaction_amount` (Fractional), `merchant_category` (String), `account_age_days` (Integral), `is_fraud` (Integral). Both `RecordIdentifierFeatureName` and `EventTimeFeatureName` must exactly match one of the FeatureDefinition names.
2. The online tier: `OnlineStoreConfig={"EnableOnlineStore": True, "StorageType": "Standard"}`. The Standard tier targets single-digit-millisecond p95 latency at $0.45/GB-month against a 5 GiB floor.
3. The offline tier: `OfflineStoreConfig={"S3StorageConfig": {"S3Uri": ...}, "DisableGlueTableCreation": False}`. The offline tier writes records as Parquet to the S3 prefix and auto-creates a Glue Data Catalog table at first ingest.

Pass `RoleArn=FG_ROLE_ARN` so Feature Store can write to the offline-tier bucket and create the Glue table. Tag the Feature Group with the lab tag so the orphan scan finds it on a failed cleanup.

After `create_feature_group` returns, poll `describe_feature_group` until `FeatureGroupStatus=Created`. Typical wait is 60 to 90 seconds while AWS provisions both tiers.


In [ ]:
# NBVAL_SKIP
# Task 2: Build the schema, create the Feature Group, poll until Created.

sagemaker = boto3.client(
    "sagemaker",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

feature_definitions = [
    {"FeatureName": "customer_id", "FeatureType": "String"},
    {"FeatureName": "event_time", "FeatureType": "Fractional"},
    {"FeatureName": "transaction_amount", "FeatureType": "Fractional"},
    {"FeatureName": "merchant_category", "FeatureType": "String"},
    {"FeatureName": "account_age_days", "FeatureType": "Integral"},
    {"FeatureName": "is_fraud", "FeatureType": "Integral"},
]

OFFLINE_S3_URI = f"s3://{BUCKET_NAME}/feature-store-offline/"

try:
    # YOUR CODE: Call sagemaker.create_feature_group with FeatureGroupName=FG_NAME,
    # RecordIdentifierFeatureName="customer_id", EventTimeFeatureName="event_time",
    # FeatureDefinitions=feature_definitions, RoleArn=FG_ROLE_ARN, Description=
    # "labrun mla lab 03 transactions feature group",
    # OnlineStoreConfig={"EnableOnlineStore": True, "StorageType": "Standard"},
    # OfflineStoreConfig={"S3StorageConfig": {"S3Uri": OFFLINE_S3_URI},
    # "DisableGlueTableCreation": False}, Tags=[{"Key": LAB_TAG_KEY,
    # "Value": LAB_TAG_VALUE}].
    pass
except ClientError as e:
    if e.response["Error"]["Code"] == "ResourceInUse":
        print(f"Feature Group {FG_NAME} already exists, reusing.")
    else:
        raise

print(f"Feature Group create requested: {FG_NAME}")
print("Provisioning both online Standard tier and offline S3 tier. This takes 60 to 90 seconds...")
print("Asking Feature Store to braid the online and offline tiers together, give it a minute...")

elapsed = 0
status = None
desc = None
while elapsed < 300:
    desc = sagemaker.describe_feature_group(FeatureGroupName=FG_NAME)
    status = desc["FeatureGroupStatus"]
    if status == "Created":
        print(f"Feature Group ready: {FG_NAME}")
        break
    if status in ("CreateFailed", "DeleteFailed"):
        reason = desc.get("FailureReason", "no reason returned")
        print(f"Feature Group entered {status}. Reason: {reason}")
        raise SystemExit(1)
    time.sleep(10)
    elapsed += 10
else:
    print(f"Feature Group did not reach Created within 5 minutes. Last status: {status}")
    raise SystemExit(1)

# Cache the Glue catalog references for the offline-tier queries in Task 4 and 5.
data_catalog = (desc or {}).get("OfflineStoreConfig", {}).get("DataCatalogConfig", {})
GLUE_DATABASE_NAME = data_catalog.get("Database")
GLUE_TABLE_NAME = data_catalog.get("TableName")
print(f"Offline-tier Glue database: {GLUE_DATABASE_NAME}")
print(f"Offline-tier Glue table: {GLUE_TABLE_NAME}")


In [ ]:
# NBVAL_SKIP
# Checkpoint 2: Feature Group exists in Status=Created with both tiers
# configured and the expected 6 FeatureDefinitions.

def checkpoint_2(session):
    from labrun_checks import CheckpointResult
    try:
        sm_client = boto3.client(
            "sagemaker",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        try:
            desc = sm_client.describe_feature_group(FeatureGroupName=FG_NAME)
        except ClientError as e:
            if e.response["Error"]["Code"] in ("ResourceNotFound", "ValidationException"):
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Feature Group {FG_NAME} does not exist. Verify create_feature_group "
                        f"was called and the name matches FG_NAME."
                    ),
                )
            return CheckpointResult(status="error", error_reason=str(e))

        status = desc.get("FeatureGroupStatus")
        if status != "Created":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Feature Group {FG_NAME} is in status {status!r}, not Created. Wait for "
                    f"the provisioning to complete and re-run."
                ),
            )

        online_cfg = desc.get("OnlineStoreConfig") or {}
        if not online_cfg.get("EnableOnlineStore"):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Feature Group {FG_NAME} has OnlineStoreConfig.EnableOnlineStore=False. "
                    f"This lab needs the online tier on so the Task 3 get_record loop has "
                    f"something to read."
                ),
            )
        storage_type = online_cfg.get("StorageType", "Standard")
        if storage_type != "Standard":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"OnlineStoreConfig.StorageType is {storage_type!r}. This lab uses "
                    f"Standard. The In-Memory tier bills $0.233/GB-hour against the same "
                    f"5 GiB floor and is out of scope here."
                ),
            )

        offline_cfg = desc.get("OfflineStoreConfig") or {}
        s3_uri = (offline_cfg.get("S3StorageConfig") or {}).get("S3Uri", "")
        if BUCKET_NAME not in s3_uri:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"OfflineStoreConfig.S3StorageConfig.S3Uri is {s3_uri!r} and does not "
                    f"reference the lab bucket {BUCKET_NAME}."
                ),
            )

        defs = desc.get("FeatureDefinitions", [])
        names = {d["FeatureName"] for d in defs}
        expected = {
            "customer_id",
            "event_time",
            "transaction_amount",
            "merchant_category",
            "account_age_days",
            "is_fraud",
        }
        missing = expected - names
        if missing:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Feature Group is missing FeatureDefinitions: {sorted(missing)}. "
                    f"Found: {sorted(names)}."
                ),
            )

        if desc.get("RecordIdentifierFeatureName") != "customer_id":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"RecordIdentifierFeatureName is "
                    f"{desc.get('RecordIdentifierFeatureName')!r}, expected 'customer_id'."
                ),
            )
        if desc.get("EventTimeFeatureName") != "event_time":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"EventTimeFeatureName is {desc.get('EventTimeFeatureName')!r}, "
                    f"expected 'event_time'."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(2, checkpoint_2)


<details><summary>Hint 1 (nudge)</summary>

The checkpoint walks the `describe_feature_group` response top-down: status, online tier, offline tier, feature definitions, record-identifier name, event-time name. The error message tells you exactly which field is wrong.

</details>

<details><summary>Hint 2 (direction)</summary>

The `create_feature_group` call has nine kwargs and order does not matter, but every one is required here. `OnlineStoreConfig={"EnableOnlineStore": True, "StorageType": "Standard"}` is the dual-tier toggle on the online side. `OfflineStoreConfig={"S3StorageConfig": {"S3Uri": OFFLINE_S3_URI}, "DisableGlueTableCreation": False}` is the offline side; `DisableGlueTableCreation=False` is what makes Feature Store create the Glue table you query in Task 4. `RoleArn=FG_ROLE_ARN` lets Feature Store assume the role from Task 1.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for the Task 2 create call:

```python
sagemaker.create_feature_group(
    FeatureGroupName=FG_NAME,
    RecordIdentifierFeatureName="customer_id",
    EventTimeFeatureName="event_time",
    FeatureDefinitions=feature_definitions,
    RoleArn=FG_ROLE_ARN,
    Description="labrun mla lab 03 transactions feature group",
    OnlineStoreConfig={"EnableOnlineStore": True, "StorageType": "Standard"},
    OfflineStoreConfig={
        "S3StorageConfig": {"S3Uri": OFFLINE_S3_URI},
        "DisableGlueTableCreation": False,
    },
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
```

</details>

**Wallet check.** Feature Group create plus a 90-second provisioning wait. The online Standard tier starts billing the moment the group reaches Created status at $0.45/GB-month against a 5 GiB-month floor: prorated to about half a cent per session at the floor. Cleanup at the bottom of this notebook tears the group down within the session window. Damage so far: under $0.01. Your coffee this morning cost about a hundred times more.

## Task 3: Ingest 100 synthetic transactions via PutRecord and measure online-tier read latency

The online tier holds the latest record per customer-id; the offline tier holds every version. To prove both, this lab generates 100 records covering 10 customer-ids (`customer-001` through `customer-010`) with 10 transactions each, all ingested via `put_record` against the Feature Store Runtime API.

After ingestion the cell measures online-tier read latency: 100 `get_record` calls against random customer-ids, wall-clock timed, p50 and p95 computed excluding the first 5 calls (warm-up exclusion is standard for client-side latency benchmarks; the first call pays connection-setup cost the rest do not).

Build it in this order:

1. Use `numpy.random.default_rng(seed=42)` for deterministic synthetic data so two runs of this cell produce identical records and the offline-tier row count in Task 5 is exactly 100.
2. For each of 100 records pick a customer-id from `customer-001` through `customer-010` in round-robin order, set `event_time` to a Fractional epoch-seconds value that increments per call, pick `merchant_category` from `{"food", "retail", "online", "travel", "other"}`, pick `transaction_amount` uniform in [5, 500], pick `account_age_days` uniform integer in [30, 3650], set `is_fraud` to 1 with probability 0.05.
3. Call `featurestore_runtime.put_record(FeatureGroupName=FG_NAME, Record=[{"FeatureName": ..., "ValueAsString": ...}, ...])` for each record. The Runtime API takes single records (not batches); the loop is what gets to 100.
4. Loop `get_record` 100 times against random customer-ids, capture wall-clock per call with `time.perf_counter()`, compute p50 and p95 across calls 5 through 100. The checkpoint relaxes the brief's 10 ms target to 50 ms because the network path from Colab to `us-east-1` adds variance the tier-internal target does not account for; the captured p95 still surfaces in Checkpoint 5's comparison metric.


In [ ]:
# NBVAL_SKIP
# Task 3: Generate 100 records, ingest via put_record, then measure
# online-tier GetRecord latency across 100 reads.

import numpy as np

featurestore_runtime = boto3.client(
    "sagemaker-featurestore-runtime",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

rng = np.random.default_rng(seed=42)
MERCHANTS = ["food", "retail", "online", "travel", "other"]
CUSTOMER_IDS = [f"customer-{i:03d}" for i in range(1, 11)]
NUM_RECORDS = 100

base_event_time = time.time()

print("Ingesting 100 synthetic transactions across 10 customers...")
for i in range(NUM_RECORDS):
    customer_id = CUSTOMER_IDS[i % 10]
    event_time_value = base_event_time + i  # monotone per record
    merchant = MERCHANTS[int(rng.integers(0, len(MERCHANTS)))]
    amount = float(rng.uniform(5.0, 500.0))
    age = int(rng.integers(30, 3650))
    fraud = int(rng.random() < 0.05)

    record = [
        {"FeatureName": "customer_id", "ValueAsString": customer_id},
        {"FeatureName": "event_time", "ValueAsString": f"{event_time_value:.3f}"},
        {"FeatureName": "transaction_amount", "ValueAsString": f"{amount:.2f}"},
        {"FeatureName": "merchant_category", "ValueAsString": merchant},
        {"FeatureName": "account_age_days", "ValueAsString": str(age)},
        {"FeatureName": "is_fraud", "ValueAsString": str(fraud)},
    ]

    # YOUR CODE: Call featurestore_runtime.put_record() with
    # FeatureGroupName=FG_NAME and Record=record.

print(f"Ingested {NUM_RECORDS} records across {len(CUSTOMER_IDS)} customer-ids.")
print()
print("Measuring online-tier read latency over 100 GetRecord calls...")
print("Asking the online tier to time itself, one record at a time...")

latencies_ms = []
for i in range(100):
    target = CUSTOMER_IDS[int(rng.integers(0, len(CUSTOMER_IDS)))]
    t0 = time.perf_counter()
    # YOUR CODE: Call featurestore_runtime.get_record() with
    # FeatureGroupName=FG_NAME and RecordIdentifierValueAsString=target.
    # Assign to resp.
    t1 = time.perf_counter()
    latencies_ms.append((t1 - t0) * 1000.0)

# Exclude first 5 calls (cold-start cost). Compute p50 and p95 over the rest.
warm = latencies_ms[5:]
warm_sorted = sorted(warm)
p50 = warm_sorted[len(warm_sorted) // 2]
p95_index = max(0, int(len(warm_sorted) * 0.95) - 1)
p95 = warm_sorted[p95_index]
ONLINE_P95_MS = p95

print()
print(f"Online-tier GetRecord p50: {p50:.1f} ms")
print(f"Online-tier GetRecord p95: {p95:.1f} ms (warm calls, excludes first 5)")
print(f"Min: {min(warm):.1f} ms, Max: {max(warm):.1f} ms over {len(warm)} warm calls")


In [ ]:
# NBVAL_SKIP
# Checkpoint 3: 10 sequential GetRecord calls (one per customer-id) all
# return records, and the measured online-tier p95 is below 50 ms.

def checkpoint_3(session):
    from labrun_checks import CheckpointResult
    try:
        fs_runtime = boto3.client(
            "sagemaker-featurestore-runtime",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        expected_ids = [f"customer-{i:03d}" for i in range(1, 11)]
        latencies = []
        for cid in expected_ids:
            t0 = time.perf_counter()
            try:
                resp = fs_runtime.get_record(
                    FeatureGroupName=FG_NAME,
                    RecordIdentifierValueAsString=cid,
                )
            except ClientError as e:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"GetRecord on {cid!r} failed with "
                        f"{e.response['Error']['Code']}. Confirm the ingestion loop ran "
                        f"for all 10 customer-ids and the online tier is enabled."
                    ),
                )
            t1 = time.perf_counter()
            latencies.append((t1 - t0) * 1000.0)

            record = resp.get("Record") or []
            if not record:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"GetRecord on {cid!r} returned an empty record. The online tier "
                        f"either has no record for this customer-id or the ingestion did "
                        f"not commit. Re-run the put_record loop."
                    ),
                )

        # Wall-clock p95 across the 10 calls (no warm-up exclusion at this size).
        sorted_lat = sorted(latencies)
        p95_index = max(0, int(len(sorted_lat) * 0.95) - 1)
        measured_p95 = sorted_lat[p95_index]
        # 50 ms ceiling: relaxes the tier-internal 10 ms target to absorb client-side
        # network jitter from Colab to us-east-1. The captured p95 surfaces in CP5.
        if measured_p95 > 50.0:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Online-tier p95 across 10 GetRecord calls measured {measured_p95:.1f} ms, "
                    f"above the 50 ms ceiling. Run the latency cell again on a less loaded "
                    f"network or re-run from a region closer to {REGION}."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(3, checkpoint_3)


<details><summary>Hint 1 (nudge)</summary>

Two failure shapes are possible: a GetRecord call returns nothing for one of the 10 customer-ids (ingestion did not cover that id), or the p95 is above 50 ms (network path is the likely cause, not the tier). The error message tells you which.

</details>

<details><summary>Hint 2 (direction)</summary>

`featurestore_runtime.put_record(FeatureGroupName=FG_NAME, Record=record)` is one call. The `Record` shape is a list of `{"FeatureName": str, "ValueAsString": str}` dicts; the cell already builds the list. `featurestore_runtime.get_record(FeatureGroupName=FG_NAME, RecordIdentifierValueAsString=target)` is the read; the `RecordIdentifierValueAsString` is a string (cast integers if you ever change customer-ids to numeric). The Runtime API has no batch_put_record method on the boto3 client; the per-record loop is the only API path.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for the two YOUR CODE blanks:

```python
# Inside the ingest loop:
featurestore_runtime.put_record(
    FeatureGroupName=FG_NAME,
    Record=record,
)

# Inside the latency-measurement loop:
resp = featurestore_runtime.get_record(
    FeatureGroupName=FG_NAME,
    RecordIdentifierValueAsString=target,
)
```

</details>

**Wallet check.** 100 PutRecord writes plus 110 GetRecord reads. Feature Store online Standard tier bills per-request units at well under a penny for this volume. Plus the 5 GiB-month floor at $0.45/GB-month is still accruing in the background: another fraction of a cent for the prorated session window. Damage so far: under $0.01. Your coffee this morning cost about a hundred times more.

## Task 4: Wait for offline-tier propagation and confirm the Athena table shows all 10 customers

Feature Store buffers offline-tier writes asynchronously. PutRecord acks fast on the online side but the Parquet rows land in the offline-tier S3 prefix 5 to 15 minutes later. The Athena table the Glue catalog auto-creates points at the Parquet prefix, so the table is empty until propagation completes. Reading the offline tier before propagation gives zero rows.

The lab handles this with a polling loop. Up to 10 minutes, every 30 seconds, it runs `SELECT COUNT(DISTINCT customer_id) FROM <database>.<table> WHERE is_deleted = false`. The first few queries return 0 (table exists, no rows yet). Once propagation lands, the count steps up to 10 (the ten distinct customer-ids the ingestion seeded).

Build it in this order:

1. Configure the Athena client with the output location `s3://{bucket}/athena-results/`. Athena writes query result CSVs there; the offline-tier bucket is the natural home for them and the cleanup cell sweeps the prefix.
2. Loop: `athena.start_query_execution` with the COUNT(DISTINCT) SQL above, poll `athena.get_query_execution` every 5 seconds until `Status.State == "SUCCEEDED"`. Read the result with `athena.get_query_results` and parse the single cell.
3. If the count is 10, print success. If the count is below 10 after 10 minutes, the offline tier propagation is running slow (rare but possible); the cell prints a directional message and the student can re-run Task 4 alone.


In [ ]:
# NBVAL_SKIP
# Task 4: Wait for offline-tier propagation, run COUNT(DISTINCT customer_id)
# against the auto-created Glue table via Athena.

athena = boto3.client(
    "athena",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

ATHENA_OUTPUT = f"s3://{BUCKET_NAME}/athena-results/"
print(f"Athena results will land at {ATHENA_OUTPUT}")
print(f"Querying offline-tier table: {GLUE_DATABASE_NAME}.{GLUE_TABLE_NAME}")
print()
print("Asking Athena to put on her reading glasses. Offline-tier propagation")
print("takes 5 to 15 minutes from PutRecord to first row visible...")


def run_athena_query(sql):
    """Submit a query, poll until SUCCEEDED, return Athena ResultSet rows."""
    # YOUR CODE: Call athena.start_query_execution with QueryString=sql,
    # QueryExecutionContext={"Database": GLUE_DATABASE_NAME},
    # ResultConfiguration={"OutputLocation": ATHENA_OUTPUT}. Assign the
    # returned dict to start_resp.

    qid = start_resp["QueryExecutionId"]
    poll_elapsed = 0
    while poll_elapsed < 120:
        info = athena.get_query_execution(QueryExecutionId=qid)
        state = info["QueryExecution"]["Status"]["State"]
        if state == "SUCCEEDED":
            break
        if state in ("FAILED", "CANCELLED"):
            reason = info["QueryExecution"]["Status"].get("StateChangeReason", "no reason")
            raise RuntimeError(f"Athena query {qid} ended in {state}: {reason}")
        time.sleep(5)
        poll_elapsed += 5
    else:
        raise RuntimeError(f"Athena query {qid} did not finish within 120 seconds.")

    results = athena.get_query_results(QueryExecutionId=qid)
    return results["ResultSet"]["Rows"]


count_sql = (
    f'SELECT COUNT(DISTINCT customer_id) AS distinct_customers '
    f'FROM "{GLUE_DATABASE_NAME}"."{GLUE_TABLE_NAME}" '
    f'WHERE is_deleted = false'
)

distinct_customers = 0
poll_minutes = 0
while poll_minutes < 10:
    rows = run_athena_query(count_sql)
    # Row 0 is the header; row 1 is the data row.
    if len(rows) >= 2:
        value = rows[1]["Data"][0].get("VarCharValue", "0")
        distinct_customers = int(value)
    print(f"  t={poll_minutes:>2}m  distinct customers visible: {distinct_customers}")
    if distinct_customers >= 10:
        break
    time.sleep(30)
    poll_minutes += 1  # roughly 30 seconds per loop, rounded up to "minutes"

if distinct_customers < 10:
    print()
    print("Offline-tier propagation has not yet shown all 10 customers after 10 minutes.")
    print("This is rare. Wait another 5 minutes and re-run THIS cell only.")
    print("Do NOT re-run Task 3, which would double-ingest.")
else:
    print()
    print(f"Offline-tier table is reading correctly. Distinct customer count: {distinct_customers}")


In [ ]:
# NBVAL_SKIP
# Checkpoint 4: independent Athena query confirms the offline-tier table
# is readable and reports 10 distinct customers.

def checkpoint_4(session):
    from labrun_checks import CheckpointResult
    try:
        athena_client = boto3.client(
            "athena",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        if not GLUE_DATABASE_NAME or not GLUE_TABLE_NAME:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Glue database/table names were not resolved by Task 2 "
                    f"(database={GLUE_DATABASE_NAME!r}, table={GLUE_TABLE_NAME!r}). "
                    f"Re-run Task 2 so describe_feature_group fills them in."
                ),
            )

        sql = (
            f'SELECT COUNT(DISTINCT customer_id) AS distinct_customers '
            f'FROM "{GLUE_DATABASE_NAME}"."{GLUE_TABLE_NAME}" '
            f'WHERE is_deleted = false'
        )
        try:
            start_resp = athena_client.start_query_execution(
                QueryString=sql,
                QueryExecutionContext={"Database": GLUE_DATABASE_NAME},
                ResultConfiguration={"OutputLocation": f"s3://{BUCKET_NAME}/athena-results/"},
            )
        except ClientError as e:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"start_query_execution failed: {e.response['Error']['Code']}. "
                    f"Confirm the Athena WorkGroup output location is set and the IAM "
                    f"caller has athena:StartQueryExecution."
                ),
            )

        qid = start_resp["QueryExecutionId"]
        elapsed = 0
        state = None
        while elapsed < 120:
            info = athena_client.get_query_execution(QueryExecutionId=qid)
            state = info["QueryExecution"]["Status"]["State"]
            if state == "SUCCEEDED":
                break
            if state in ("FAILED", "CANCELLED"):
                reason = info["QueryExecution"]["Status"].get("StateChangeReason", "")
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Athena query for the distinct-customer count ended in {state}. "
                        f"Reason: {reason!r}."
                    ),
                )
            time.sleep(5)
            elapsed += 5
        if state != "SUCCEEDED":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Athena query did not finish within 120 seconds (last state {state!r})."
                ),
            )

        results = athena_client.get_query_results(QueryExecutionId=qid)
        rows = results["ResultSet"]["Rows"]
        if len(rows) < 2:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Athena returned no data row. The offline tier may still be propagating; "
                    f"wait 5 minutes and re-run."
                ),
            )
        value = rows[1]["Data"][0].get("VarCharValue", "0")
        distinct = int(value)
        if distinct != 10:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Athena reports {distinct} distinct customers in the offline tier, "
                    f"expected 10. If the count is below 10 the offline-tier propagation "
                    f"has not finished (wait 5 minutes and re-run). If the count is above "
                    f"10 the ingest loop covered more customer-ids than the schema expects."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(4, checkpoint_4)


<details><summary>Hint 1 (nudge)</summary>

The most common failure mode here is patience. The offline tier really does take 5 to 15 minutes after the last PutRecord. The polling loop in the cell handles up to 10 minutes; if it times out, wait 5 minutes outside the notebook and re-run THIS cell only. Do not re-run Task 3.

</details>

<details><summary>Hint 2 (direction)</summary>

The `run_athena_query` helper takes a SQL string and returns the `ResultSet.Rows` list. You only need to fill in the `start_query_execution` call. Three kwargs: `QueryString=sql` (passed in), `QueryExecutionContext={"Database": GLUE_DATABASE_NAME}` (the catalog database name from Task 2), `ResultConfiguration={"OutputLocation": ATHENA_OUTPUT}` (the S3 prefix the cell already declared).

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for the YOUR CODE blank inside `run_athena_query`:

```python
start_resp = athena.start_query_execution(
    QueryString=sql,
    QueryExecutionContext={"Database": GLUE_DATABASE_NAME},
    ResultConfiguration={"OutputLocation": ATHENA_OUTPUT},
)
```

</details>

**Wallet check.** Athena bills $5 per TB scanned with a 10 MB minimum per query. Two queries against a Parquet table holding about 50 KB of data round up to a 10 MB scan each: 20 MB total at $5/TB rounds to fractions of a penny. Plus the offline-tier S3 storage is fractions of a penny at 50 KB. Damage so far: still under $0.02. Your coffee this morning cost about fifty times more.

## Task 5: Confirm the offline tier returns the full versioned history

The point of the offline tier is that every PutRecord becomes a new row. The online tier returns only the latest record per customer-id; the offline tier returns all of them. To prove this, run an Athena query that groups by `customer_id` and counts rows: each of the 10 customers should report exactly 10 versions, because the ingest loop in Task 3 covered 10 customers with 10 transactions each.

This is the exam-relevant point of the lab. The online tier is the inference path: low-latency, single-row, latest-version-wins. The offline tier is the training path: append-only, full history, queryable as a fact table. One Feature Group definition, two read patterns.

The cell runs the GROUP BY query, prints the per-customer version counts, and Checkpoint 5 surfaces the captured offline row count next to the online p95 from Checkpoint 3 as the lab's comparison metric.


In [ ]:
# NBVAL_SKIP
# Task 5: Query the offline tier with GROUP BY customer_id and confirm each
# customer has exactly 10 versions in the append-only history.

version_sql = (
    f'SELECT customer_id, COUNT(*) AS version_count '
    f'FROM "{GLUE_DATABASE_NAME}"."{GLUE_TABLE_NAME}" '
    f'WHERE is_deleted = false '
    f'GROUP BY customer_id '
    f'ORDER BY customer_id'
)

# YOUR CODE: Call athena.start_query_execution() with QueryString=version_sql,
# QueryExecutionContext={"Database": GLUE_DATABASE_NAME}, and
# ResultConfiguration={"OutputLocation": f"s3://{BUCKET_NAME}/athena-results/"}.
# Assign to start_resp.

qid = start_resp["QueryExecutionId"]
elapsed = 0
while elapsed < 120:
    info = athena.get_query_execution(QueryExecutionId=qid)
    state = info["QueryExecution"]["Status"]["State"]
    if state == "SUCCEEDED":
        break
    if state in ("FAILED", "CANCELLED"):
        reason = info["QueryExecution"]["Status"].get("StateChangeReason", "")
        print(f"Athena GROUP BY query ended in {state}: {reason}")
        raise SystemExit(1)
    time.sleep(5)
    elapsed += 5

results = athena.get_query_results(QueryExecutionId=qid)
rows = results["ResultSet"]["Rows"]

print(f"Offline-tier version count per customer (from {GLUE_TABLE_NAME}):")
total_rows = 0
per_customer = []
for row in rows[1:]:  # skip header
    data = row["Data"]
    cid = data[0].get("VarCharValue", "")
    vcount = int(data[1].get("VarCharValue", "0"))
    per_customer.append((cid, vcount))
    total_rows += vcount
    print(f"  {cid}: {vcount} versions")

OFFLINE_ROW_COUNT = total_rows
print()
print(f"Total rows across all customers: {OFFLINE_ROW_COUNT}")
print(f"Online tier returns 1 row per customer ({len(per_customer)} total).")
print(f"Offline tier returns the full history ({OFFLINE_ROW_COUNT} rows across the same {len(per_customer)} customers).")


In [ ]:
# NBVAL_SKIP
# Checkpoint 5: independent Athena GROUP BY query confirms 10 customers each
# with exactly 10 versions = 100 total rows. Surfaces the online p95 next to
# the offline row count as the comparison metric (lib/labs.ts reads it).

def checkpoint_5(session):
    from labrun_checks import CheckpointResult
    try:
        athena_client = boto3.client(
            "athena",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        sql = (
            f'SELECT customer_id, COUNT(*) AS version_count '
            f'FROM "{GLUE_DATABASE_NAME}"."{GLUE_TABLE_NAME}" '
            f'WHERE is_deleted = false '
            f'GROUP BY customer_id'
        )
        try:
            start_resp = athena_client.start_query_execution(
                QueryString=sql,
                QueryExecutionContext={"Database": GLUE_DATABASE_NAME},
                ResultConfiguration={"OutputLocation": f"s3://{BUCKET_NAME}/athena-results/"},
            )
        except ClientError as e:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"start_query_execution failed for the GROUP BY query: "
                    f"{e.response['Error']['Code']}."
                ),
            )
        qid = start_resp["QueryExecutionId"]
        elapsed = 0
        state = None
        while elapsed < 120:
            info = athena_client.get_query_execution(QueryExecutionId=qid)
            state = info["QueryExecution"]["Status"]["State"]
            if state == "SUCCEEDED":
                break
            if state in ("FAILED", "CANCELLED"):
                reason = info["QueryExecution"]["Status"].get("StateChangeReason", "")
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Athena GROUP BY query ended in {state}: {reason!r}."
                    ),
                )
            time.sleep(5)
            elapsed += 5
        if state != "SUCCEEDED":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Athena GROUP BY query did not finish within 120 seconds "
                    f"(last state {state!r})."
                ),
            )

        results = athena_client.get_query_results(QueryExecutionId=qid)
        rows = results["ResultSet"]["Rows"]
        data_rows = rows[1:] if rows else []
        if len(data_rows) != 10:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"GROUP BY returned {len(data_rows)} customer rows, expected 10. "
                    f"Either the offline tier is still propagating or the ingest loop "
                    f"did not cover 10 distinct customer-ids."
                ),
            )

        per_customer = {}
        for row in data_rows:
            data = row["Data"]
            cid = data[0].get("VarCharValue", "")
            vcount = int(data[1].get("VarCharValue", "0"))
            per_customer[cid] = vcount

        bad = {cid: v for cid, v in per_customer.items() if v != 10}
        if bad:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Expected exactly 10 versions per customer (100 rows total). "
                    f"Customers with the wrong count: {bad}."
                ),
            )

        # Comparison metric: online p95 next to offline row count, surfaced in the
        # Option D Colab card on PASS per blueprint Section 21.
        online_p95_str = f"{ONLINE_P95_MS:.1f} ms" if ONLINE_P95_MS is not None else "not captured"
        total_rows = sum(per_customer.values())
        print()
        print(
            f"Comparison metric: online tier p95 = {online_p95_str} across 100 reads; "
            f"offline tier returns {total_rows} rows across {len(per_customer)} customers via Athena."
        )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(5, checkpoint_5)


<details><summary>Hint 1 (nudge)</summary>

This is the same Athena pattern as Task 4, swapped to GROUP BY. If the failure is "expected 10 rows, got N," the offline tier still has not propagated all 100 records; wait a few minutes and re-run.

</details>

<details><summary>Hint 2 (direction)</summary>

The `start_query_execution` call takes the same three kwargs as Task 4: `QueryString=version_sql`, `QueryExecutionContext={"Database": GLUE_DATABASE_NAME}`, and `ResultConfiguration={"OutputLocation": f"s3://{BUCKET_NAME}/athena-results/"}`. The poll loop and result parsing are already written below your blank.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for the YOUR CODE blank:

```python
start_resp = athena.start_query_execution(
    QueryString=version_sql,
    QueryExecutionContext={"Database": GLUE_DATABASE_NAME},
    ResultConfiguration={"OutputLocation": f"s3://{BUCKET_NAME}/athena-results/"},
)
```

</details>

**Wallet check.** One more Athena query at the 10 MB minimum scan. Still well under a penny. The biggest line item this session is the Feature Store online Standard tier prorated to about half a cent at the 5 GiB-month floor. Damage so far: about $0.02 to $0.03. Your coffee this morning cost about thirty times more, and you now know which tier of Feature Store handles which read pattern.

## Cleanup

The Feature Group teardown is the longest single step here. `delete_feature_group` returns immediately but the underlying online-tier teardown takes 1 to 2 minutes. The cleanup cell polls `describe_feature_group` until it returns `ResourceNotFound` before declaring the group gone. The Glue Data Catalog table the Feature Store auto-created is deleted as part of the Feature Group teardown; the offline-tier S3 objects are NOT (the bucket and the records under `feature-store-offline/` and `athena-results/` survive the delete unless you sweep them).

The cell sweeps the bucket manually, then runs `run_cleanup` against the adapter-supported IAM role and S3 bucket entries. Re-running the cell is safe.


In [ ]:
# NBVAL_SKIP
# Cleanup. Manual Feature Group delete first (no adapter in labrun-checks 0.6.0),
# then sweep the bucket, then run_cleanup walks the IAM and S3 entries.

sagemaker_cleanup = boto3.client(
    "sagemaker",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
s3_cleanup = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

manual_warnings = []
fg_destroyed = False

try:
    sagemaker_cleanup.delete_feature_group(FeatureGroupName=FG_NAME)
    print(f"Requested delete on Feature Group {FG_NAME}.")
    print("Waiting for the online tier to tear down (1 to 2 minutes typical)...")
    print("Telling the 5 GiB-month floor to stop accruing rent...")

    elapsed = 0
    while elapsed < 300:
        try:
            sagemaker_cleanup.describe_feature_group(FeatureGroupName=FG_NAME)
            time.sleep(15)
            elapsed += 15
        except ClientError as e:
            if e.response["Error"]["Code"] in ("ResourceNotFound", "ValidationException"):
                print(f"Feature Group {FG_NAME} is gone. The online tier floor has stopped billing.")
                fg_destroyed = True
                break
            raise
    if not fg_destroyed:
        manual_warnings.append(
            f"FAILED TO DELETE: sagemaker_feature_group {FG_NAME!r} did not disappear within "
            f"5 minutes. Run manually: aws sagemaker delete-feature-group "
            f"--feature-group-name {FG_NAME}"
        )
except ClientError as e:
    if e.response["Error"]["Code"] in ("ResourceNotFound", "ValidationException"):
        print(f"Feature Group {FG_NAME} already gone, skipping.")
        fg_destroyed = True
    else:
        manual_warnings.append(
            f"FAILED TO DELETE: sagemaker_feature_group {FG_NAME!r}. Error: {e}. "
            f"Run manually: aws sagemaker delete-feature-group --feature-group-name {FG_NAME}"
        )


def _empty_bucket(bucket_name):
    """Delete every object in the bucket so s3 rb succeeds."""
    paginator = s3_cleanup.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=bucket_name):
        keys = [{"Key": obj["Key"]} for obj in page.get("Contents", [])]
        if keys:
            s3_cleanup.delete_objects(Bucket=bucket_name, Delete={"Objects": keys})


try:
    _empty_bucket(BUCKET_NAME)
    print(f"Emptied bucket {BUCKET_NAME} of all offline-tier records and Athena results.")
except ClientError as e:
    if e.response["Error"]["Code"] in ("NoSuchBucket", "404"):
        print(f"Bucket {BUCKET_NAME} already gone.")
    else:
        manual_warnings.append(
            f"FAILED TO EMPTY: bucket {BUCKET_NAME!r}. Error: {e}. "
            f"Run manually: aws s3 rm s3://{BUCKET_NAME} --recursive"
        )

result = run_cleanup(CLEANUP_MANIFEST)

for warning in manual_warnings:
    print(warning)
for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if manual_warnings or result.warnings or result.orphans:
    print()

failed_ids = set()
for w in result.warnings:
    for r in CLEANUP_MANIFEST:
        if r.id in w:
            failed_ids.add(r.id)
            break

# Accounting:
# - critical: the Feature Group (online-tier floor billing risk). 1 total.
# - standard: everything in CLEANUP_MANIFEST (IAM role + S3 bucket). 2 total.
critical_total = 1
critical_destroyed = 1 if fg_destroyed else 0
standard_total = len(CLEANUP_MANIFEST)
standard_destroyed = standard_total - len(failed_ids)
failed_count = len(failed_ids) + len(manual_warnings)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed} of {critical_total}")
print(f"Standard resources destroyed: {standard_destroyed} of {standard_total}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your AWS account may still incur charges. Resolve before closing this notebook.")

final_status = "clean" if (failed_count == 0 and result.status == "clean" and fg_destroyed) else "dirty"
cleanup(status=final_status)

if failed_count > 0 or not fg_destroyed:
    sys.exit(1)


**Session total: about $0.02 to $0.03.** Feature Store online Standard tier at the 5 GiB-month floor prorated to about half a cent for the session. Per-request units across 100 writes and 110 reads sat in the fractions-of-a-cent range. Offline-tier S3 storage at 50 KB was effectively free. Two Athena queries at the 10 MB minimum scan each cost fractions of a penny. The 5 GiB-month floor stopped billing the moment the Feature Group teardown completed. Check your AWS Billing console in 24 hours to confirm zero ongoing charges.

## Reflection

These are not graded. They are for you.

1. Walk through which AWS service serves which read in this lab. The real-time fraud scorer hits the online tier at single-digit-millisecond p95; the monthly retrainer hits the offline tier via Athena over a 90-day window. Where do these reads physically live, and why is the online tier so much faster? What would change if you used the In-Memory tier instead of Standard, and what does the cost math look like at 5 GiB versus 100 GiB of online state?

2. The offline tier is append-only: every PutRecord adds a new row, nothing updates in place. Walk through why that design is the right one for ML training (think about time-travel queries and reproducibility). What problems does it cause for storage growth at 1 billion records per day, and how do you address them?

## Exam-Style Practice

**Question 1 (Domain 1, online vs offline tier selection):**

A real-time recommendation service needs feature lookups at p95 below 5 ms during inference. A separate batch training pipeline needs to query 90 days of historical feature snapshots for monthly retraining. The team wants both reads served from a single source of truth. Which SageMaker Feature Store configuration fits?

A. Create a Feature Group with `OnlineStoreConfig.EnableOnlineStore=True` (Standard tier) and `OfflineStoreConfig.S3StorageConfig.S3Uri` set. Real-time lookups go to the online tier; the batch pipeline queries the offline tier via Athena.

B. Create two separate Feature Groups: one online-only for real-time, one offline-only for batch. Sync them via a daily Lambda.

C. Create a Feature Group with only `OfflineStoreConfig` set; cache the latest record per customer in ElastiCache for low-latency reads.

D. Skip Feature Store entirely; store features in DynamoDB for online reads and in S3 Parquet for offline reads.

<details><summary>Show answer</summary>

**Correct: A.**

- A is correct: SageMaker Feature Store is purpose-built for this exact split. A single Feature Group can carry both an online tier (low-latency single-record lookups, GetRecord API) and an offline tier (append-only history in S3, queryable via Athena and the auto-created Glue table). The two tiers stay consistent because the same ingest path writes both.
- B is wrong: maintaining two Feature Groups in sync introduces consistency bugs and doubles the operational surface. Feature Store exists to avoid this pattern.
- C is wrong: an offline-only Feature Group plus ElastiCache means writing custom sync logic to push records to Redis. Feature Store online tier removes that requirement entirely.
- D is wrong: DynamoDB plus S3 Parquet works but loses the AWS-managed schema enforcement, time-travel semantics, and integration with SageMaker Training and Pipelines that Feature Store provides.

</details>

**Question 2 (Domain 1, online tier storage type selection; constraint-driven comparison per blueprint Section 21):**

A team is configuring a Feature Store online tier and is choosing between `StorageType=Standard` and `StorageType=InMemory`. Their workload requires p99 read latency below 5 ms and serves 50,000 reads per second across 500 GiB of online state. Cost is a constraint. Which configuration fits?

A. `StorageType=InMemory`, sized for 500 GiB. Cost at $0.233/GB-hour = ~$84,000/month.

B. `StorageType=Standard`, sized for 500 GiB. Cost at $0.45/GB-month = ~$225/month plus per-request units.

C. `StorageType=Standard` for the 5 ms requirement; provision additional read units to handle the 50K-rps load.

D. `StorageType=InMemory` for the latency requirement; AWS does not bill the In-Memory tier against a storage floor at this volume.

<details><summary>Show answer</summary>

**Correct: A.**

- A is correct: the In-Memory tier publishes sub-millisecond p99 latency and supports the 50K-rps throughput requirement. At 500 GiB it costs ~$84,000/month, which is the price of the latency requirement. Standard tier's published p95 is "single-digit milliseconds" (10 ms typical) and may not meet a 5 ms p99 SLA.
- B is wrong on the latency requirement: Standard tier targets ~10 ms p95, not 5 ms p99. The cost is much lower but the SLA is unmet.
- C is wrong: provisioning more read units on Standard does not reduce per-record read latency. Read units control throughput (rps), not per-request latency. The tier choice (Standard vs InMemory) is what determines latency.
- D is wrong on the floor: the In-Memory tier DOES bill against a 5 GiB-month floor like Standard. At 500 GiB the floor is irrelevant (500 is far above 5) but the claim that it does not exist is false.

</details>
